In [14]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_log_error
from statsmodels.tsa.deterministic import DeterministicProcess

In [15]:
# path to the dataset in Kaggle's notebook
# Change to the path that stores your files
path = '../input/store-sales-time-series-forecasting/'

### 1. Compute Moving Average of Oil Prices

In [16]:
# read oil price
data_oil = pd.read_csv(path + 'oil.csv', parse_dates=['date'], infer_datetime_format=True, index_col='date')

########################################################################################################################
# TODO: compute data_oil['ma_oil'] as the moving average of data_oil['dcoilwtico'] with window size 7
# Hint: check the documentation of .rolling() method of pandas.DataFrame
########################################################################################################################
data_oil['ma_oil'] = data_oil['dcoilwtico'].rolling(7).mean()


# Create continguous moving average of oil prices
calendar = pd.DataFrame(index=pd.date_range('2013-01-01', '2017-08-31'))

#data_oil.head(50)

In [17]:
########################################################################################################################
# TODO 1: merge two DataFrame instances (data_oil and calendar) such that the merged instances has the same indexes
# as calendar.
# TODO 2: replace each NaN in data_oil['ma_oil'] by the first non-null value before it.
# Hint: check the documentation of .merge() and .fillna() methods of pandas.DataFrame
########################################################################################################################
calendar = calendar.merge(data_oil, how='outer', left_index=True, right_index=True)
calendar['ma_oil'] = calendar['ma_oil'].fillna(method="ffill")

calendar.head(15)

,dcoilwtico,ma_oil
2013-01-01,NaN,NaN
2013-01-02,93.14,NaN
2013-01-03,92.97,NaN
2013-01-04,93.12,NaN
2013-01-05,NaN,NaN
2013-01-06,NaN,NaN
2013-01-07,93.20,NaN
2013-01-08,93.21,NaN
2013-01-09,93.08,NaN
2013-01-10,93.81,93.218571


### 2. Create Workday Feature

In [18]:
########################################################################################################################
# TODO: create a True/False feature calendar['wd'] to indicate whether each date is a workday (Monday-Friday) or not.
# Hint: check documentation of pandas.DatetimeIndex.dayofweek
########################################################################################################################
calendar['wd'] = calendar.index.dayofweek < 5
calendar.head(15) # display some entries of calendar

,dcoilwtico,ma_oil,wd
2013-01-01,NaN,NaN,True
2013-01-02,93.14,NaN,True
2013-01-03,92.97,NaN,True
2013-01-04,93.12,NaN,True
2013-01-05,NaN,NaN,False
2013-01-06,NaN,NaN,False
2013-01-07,93.20,NaN,True
2013-01-08,93.21,NaN,True
2013-01-09,93.08,NaN,True
2013-01-10,93.81,93.218571,True


### 3. Read Train and Test Data

In [19]:
df_train = pd.read_csv(path + 'train.csv',
                       usecols=['store_nbr', 'family', 'date', 'sales'],
                       dtype={'store_nbr': 'category', 'family': 'category', 'sales': 'float32'},
                       parse_dates=['date'], infer_datetime_format=True)

df_train.date = df_train.date.dt.to_period('D')
df_train = df_train.set_index(['store_nbr', 'family', 'date']).sort_index()

df_train.head(15) # display some entries of the training data

sales
store_nbr family     date             
1         AUTOMOTIVE 2013-01-01    0.0
                     2013-01-02    2.0
                     2013-01-03    3.0
                     2013-01-04    3.0
                     2013-01-05    5.0
                     2013-01-06    2.0
                     2013-01-07    0.0
                     2013-01-08    2.0
                     2013-01-09    2.0
                     2013-01-10    2.0
                     2013-01-11    3.0
                     2013-01-12    2.0
                     2013-01-13    2.0
                     2013-01-14    2.0
                     2013-01-15    1.0

In [20]:
df_test = pd.read_csv(path + 'test.csv',
                      usecols=['store_nbr', 'family', 'date'],
                      dtype={'store_nbr': 'category', 'family': 'category'},
                      parse_dates=['date'], infer_datetime_format=True)

df_test.date = df_test.date.dt.to_period('D')
df_test = df_test.set_index(['store_nbr', 'family', 'date']).sort_index()

df_test.head(15) # display some entries of the testing data

Empty DataFrame
Columns: []
Index: [(1, AUTOMOTIVE, 2017-08-16), (1, AUTOMOTIVE, 2017-08-17), (1, AUTOMOTIVE, 2017-08-18), (1, AUTOMOTIVE, 2017-08-19), (1, AUTOMOTIVE, 2017-08-20), (1, AUTOMOTIVE, 2017-08-21), (1, AUTOMOTIVE, 2017-08-22), (1, AUTOMOTIVE, 2017-08-23), (1, AUTOMOTIVE, 2017-08-24), (1, AUTOMOTIVE, 2017-08-25), (1, AUTOMOTIVE, 2017-08-26), (1, AUTOMOTIVE, 2017-08-27), (1, AUTOMOTIVE, 2017-08-28), (1, AUTOMOTIVE, 2017-08-29), (1, AUTOMOTIVE, 2017-08-30)]

In [21]:
# set the range of data used in training
sdate = '2017-04-01'
edate = '2017-08-15'

# we will train a model that takes feature of a date as input and predicts the sales for each store and family of goods on that date.
y = df_train.unstack(['store_nbr', 'family']).loc[sdate:edate]

########################################################################################################################
# TODO: create the trend feature X: the value for sdate is 1, the value for the next day of sdate is 2, etc.
# Hint: check the documentation of DeterministicProcess, or this tutorial: https://www.kaggle.com/code/ryanholbrook/trend.
########################################################################################################################
dp = DeterministicProcess(index=y.index, constant=True, order=1, drop=True) # change 'None' to your answer
X = dp.in_sample()

# Extentions
X['oil']  = calendar.loc[sdate:edate]['ma_oil'].values
X['wd']   = calendar.loc[sdate:edate]['wd'].values

X.head(15)

,const,trend,oil,wd
date,,,,
2017-04-01,1.0,1.0,48.570000,False
2017-04-02,1.0,2.0,48.570000,False
2017-04-03,1.0,3.0,49.034286,True
2017-04-04,1.0,4.0,49.561429,True
2017-04-05,1.0,5.0,50.150000,True
2017-04-06,1.0,6.0,50.625714,True
2017-04-07,1.0,7.0,51.022857,True
2017-04-08,1.0,8.0,51.022857,False
2017-04-09,1.0,9.0,51.022857,False


### 4. Train Model!

In [22]:
model = LinearRegression()
model.fit(X, y)
y_pred = pd.DataFrame(model.predict(X), index=X.index, columns=y.columns)
y_pred

sales                                                          \
store_nbr           1                                                           
family     AUTOMOTIVE BABY CARE    BEAUTY    BEVERAGES     BOOKS BREAD/BAKERY   
date                                                                            
2017-04-01   3.565055       0.0  2.424179  1838.727570  0.385805   311.411773   
2017-04-02   3.568308       0.0  2.428772  1837.271355  0.382164   310.790968   
2017-04-03   4.437861       0.0  2.873867  2374.902240  0.639175   413.209379   
2017-04-04   4.448270       0.0  2.794831  2359.490010  0.628227   409.156502   
2017-04-05   4.459513       0.0  2.706050  2342.451469  0.616428   404.703682   
...               ...       ...       ...          ...       ...          ...   
2017-08-11   4.862112       0.0  3.454141  2182.795529  0.164393   331.816443   
2017-08-12   4.005367       0.0  2.944575  1629.960332 -0.106336   225.133594   
2017-08-13   4.008619       0.0  2.949168  1628.504117 -0.109977   224.512789   
2017-08-14   4.869077       0.0  3.500554  2183.873134  0.156322   331.293373   
2017-08-15   4.868451       0.0  3.550474  2189.981155  0.156641   332.532769   

                                                            ...            \
store_nbr                                                   ...         9   
family     CELEBRATION    CLEANING       DAIRY        DELI  ... MAGAZINES   
date                                                        ...             
2017-04-01   10.949809  486.815984  648.140226  115.933924  ...  4.944980   
2017-04-02   10.903122  485.894543  646.767046  115.753770  ...  4.939442   
2017-04-03   20.208190  784.289373  839.983364  152.523558  ...  3.937069   
2017-04-04   19.776953  774.839603  832.468857  151.394326  ...  3.997935   
2017-04-05   19.300904  764.396016  824.238693  150.154496  ...  4.066539   
...                ...         ...         ...         ...  ...       ...   
2017-08-11   14.061870  662.791749  660.238351  128.913259  ...  3.230448   
2017-08-12    4.324735  355.042634  458.866643   90.947254  ...  4.280231   
2017-08-13    4.278049  354.121193  457.493463   90.767101  ...  4.274693   
2017-08-14   14.071880  663.355554  658.515426  128.743170  ...  3.187920   
2017-08-15   14.233623  667.056514  660.470879  129.077422  ...  3.146391   

                                                                           \
store_nbr                                                                   
family           MEATS PERSONAL CARE PET SUPPLIES PLAYERS AND ELECTRONICS   
date                                                                        
2017-04-01  454.746469    708.204738    10.559584               19.302452   
2017-04-02  454.435141    707.439429    10.559345               19.243629   
2017-04-03  392.059505    471.435252     6.890601               14.022627   
2017-04-04  390.412543    472.267746     6.884568               14.176083   
2017-04-05  388.609937    473.286434     6.877860               14.354275   
...                ...           ...          ...                     ...   
2017-08-11  351.319047    372.265474     6.858326                6.418291   
2017-08-12  411.895655    608.146313    10.521488               11.708614   
2017-08-13  411.584328    607.381003    10.521249               11.649792   
2017-08-14  350.906287    369.346013     6.859869                6.158984   
2017-08-15  351.318880    367.714686     6.862770                5.985105   

                                                                               \
store_nbr                                                                       
family         POULTRY PREPARED FOODS      PRODUCE SCHOOL AND OFFICE SUPPLIES   
date                                                                            
2017-04-01  592.046754     166.856344  2058.753131                 -42.755265   
2017-04-02  591.913264     166.647156  2058.002379                 -41.683694   
20

In [23]:
# Results on the training set

y_pred   = y_pred.stack(['store_nbr', 'family']).reset_index()
y_target = y.stack(['store_nbr', 'family']).reset_index().copy()

y_target['sales_pred'] = y_pred['sales'].clip(0.) # Sales should be >= 0

y_target

,date,store_nbr,family,sales,sales_pred
0,2017-04-01,1,AUTOMOTIVE,9.000000,3.565055
1,2017-04-01,1,BABY CARE,0.000000,0.000000
2,2017-04-01,1,BEAUTY,1.000000,2.424179
3,2017-04-01,1,BEVERAGES,3229.000000,1838.727570
4,2017-04-01,1,BOOKS,0.000000,0.385805
...,...,...,...,...,...
244129,2017-08-15,9,POULTRY,438.132996,378.149656
244130,2017-08-15,9,PREPARED FOODS,154.552994,91.357329
244131,2017-08-15,9,PRODUCE,2419.729004,1473.257546
244132,2017-08-15,9,SCHOOL AND OFFICE SUPPLIES,121.000000,100.039680


In [24]:
########################################################################################################################
# TODO: show the training loss for each type of product.
# Hint: check the documentation of DataFrame.groupby() and GroupBy.apply().
########################################################################################################################
y_target.groupby('family').apply(lambda r: mean_squared_log_error(r['sales'], r['sales_pred'])).sort_values(ascending=False)

family
SCHOOL AND OFFICE SUPPLIES    1.487916
LIQUOR,WINE,BEER              0.790037
LINGERIE                      0.422315
GROCERY II                    0.373721
CELEBRATION                   0.339573
LAWN AND GARDEN               0.300818
HARDWARE                      0.291368
SEAFOOD                       0.287336
BEAUTY                        0.286778
HOME AND KITCHEN I            0.278903
LADIESWEAR                    0.278059
AUTOMOTIVE                    0.275179
MAGAZINES                     0.274707
PLAYERS AND ELECTRONICS       0.238433
HOME AND KITCHEN II           0.233429
PRODUCE                       0.230674
PET SUPPLIES                  0.222810
GROCERY I                     0.216049
CLEANING                      0.215838
BEVERAGES                     0.205483
EGGS                          0.203725
FROZEN FOODS                  0.183883
MEATS                         0.175813
HOME APPLIANCES               0.162268
POULTRY                       0.157990
PERSONAL CARE     

By sorting the loss of different product families, found that the product of family 'SCHOOL AND OFFICE SUPPLIES' has the biggest loss, which means the model has the most difficulty in predicting its sales.

In [25]:
# Test predictions

stest = '2017-08-16'
etest = '2017-08-31'

########################################################################################################################
# TODO: create the feature matrix of test data.
# Hint: check the documentation of DeterministicProcess.
########################################################################################################################
X_test = dp.out_of_sample(steps = 16) # from 8.16 - 8.31

X_test['oil']  = calendar.loc[stest:etest]['ma_oil'].values 
X_test['wd']   = calendar.loc[stest:etest]['wd'].values
print(X_test)

sales_pred = pd.DataFrame(model.predict(X_test), index=X_test.index, columns=y.columns)
sales_pred = sales_pred.stack(['store_nbr', 'family'])

sales_pred[sales_pred < 0] = 0. # Sales should be >= 0

            const  trend        oil     wd
2017-08-16    1.0  138.0  48.281429   True
2017-08-17    1.0  139.0  47.995714   True
2017-08-18    1.0  140.0  47.852857   True
2017-08-19    1.0  141.0  47.852857  False
2017-08-20    1.0  142.0  47.852857  False
2017-08-21    1.0  143.0  47.688571   True
2017-08-22    1.0  144.0  47.522857   True
2017-08-23    1.0  145.0  47.645714   True
2017-08-24    1.0  146.0  47.598571   True
2017-08-25    1.0  147.0  47.720000   True
2017-08-26    1.0  148.0  47.720000  False
2017-08-27    1.0  149.0  47.720000  False
2017-08-28    1.0  150.0  47.624286   True
2017-08-29    1.0  151.0  47.320000   True
2017-08-30    1.0  152.0  47.115714   True
2017-08-31    1.0  153.0  47.060000   True


In [26]:
# Create submission

df_sub = pd.read_csv(path + 'sample_submission.csv', index_col='id')
df_sub.sales = sales_pred.values
df_sub.to_csv('submission.csv', index=True)